<a href="https://colab.research.google.com/github/Shubham4521/Cost-Aware-LLM-Router/blob/main/notebooks/00_setup_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai tavily-python

In [ ]:
import time
from google.colab import userdata
from openai import OpenAI
from tavily import TavilyClient

groq = OpenAI(base_url="https://api.groq.com/openai/v1",
              api_key=userdata.get("GROQ_API_KEY"))
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1",
                    api_key=userdata.get("OPENROUTER_API_KEY"))
tavily = TavilyClient(api_key=userdata.get("TAVILY_API_KEY"))

print("Connected")

Connected


In [ ]:
print("=== GROQ MODELS (free tier) ===")
for m in groq.models.list().data:
    print(m.id)

print("\n=== OPENROUTER MODELS (cheap + premium) ===")
for m in openrouter.models.list().data:
    if any(p in m.id for p in ["openai/", "anthropic/", "google/"]):
        print(m.id)

=== GROQ MODELS (free tier) ===
whisper-large-v3
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
allam-2-7b
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m

=== OPENROUTER MODELS (cheap + premium) ===
google/gemini-4-neon
google/gemini-4-neon:batch
openai/gpt-6.2-astra-pro
openai/gpt-6.2-astra-pro:batch
openai/gpt-6.2-astra
openai/gpt-6.2-astra:batch
openai/gpt-6.2-sol-pro
openai/gpt-6.2-sol-pro:batch
openai/gpt-6.2-sol
openai/gpt-6.2-sol:batch
google/gemini-4-hydrogen
google/gemini-4-hydrogen:batch
google/gemini-4-carbon
google/gemini-4-carbon:batch
anthropic/claude-fable-5.2
anthropic/claude-fable-5.2:batch
google/gemini-omni-1.2-flash
google/gemini-omni-1.1-flash
openai/gpt-6.1-terra-pro
openai/gpt-6.1-terra-pro:batch
openai/gpt-6.1-terra
openai/gpt-6.1-terra:batch
google/gemini-4-argon
google/gemini-4-argon:batch
anthropic/claude-haiku-5.

In [ ]:
TIERS = {
    "free":    (groq,       "openai/gpt-oss-120b"),
    "cheap":   (openrouter, "openai/gpt-5-mini"),
    "premium": (openrouter, "anthropic/claude-sonnet-5"),
}

def call(tier, prompt):
    client, model = TIERS[tier]
    start = time.time()
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000,
    )
    return {
        "tier": tier,
        "model": model,
        "answer": r.choices[0].message.content,
        "input_tokens": r.usage.prompt_tokens,
        "output_tokens": r.usage.completion_tokens,
        "latency_s": round(time.time() - start, 2),
    }

prompt = "What is 17 * 23? Answer with just the number."
for tier in TIERS:
    res = call(tier, prompt)
    print(res["tier"], "|", res["model"], "|", res["answer"].strip(),
          "|", res["input_tokens"], "+", res["output_tokens"], "tokens |",
          res["latency_s"], "s")

free | openai/gpt-oss-120b | 391 | 85 + 52 tokens | 0.43 s
cheap | openai/gpt-5-mini | 391 | 20 + 47 tokens | 2.6 s
premium | anthropic/claude-sonnet-5 | 391

17 × 23 = 391 | 23 + 13 tokens | 0.93 s


In [ ]:
r = tavily.search("top news today", max_results=3)
for item in r["results"]:
    print(item["title"], "-", item["url"])

Breaking News, Latest News and Videos | CNN - https://www.cnn.com
NBC News - Breaking Headlines and Video Reports on World, U.S. and Local Angles | NBC News - https://www.nbcnews.com
Fox News - Breaking News Updates | Latest News Headlines | Photos & News Videos - https://www.foxnews.com
